### 1. colab 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/cifar100/cifar100.zip /content/
!unzip -q /content/cifar100.zip -d /content/dataset/

Mounted at /content/drive


### 2. CIFAR100 data로 train, test dataset,loader 만들기

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torchsummary import summary
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dir = 'dataset/cifar100/train'
test_dir = 'dataset/cifar100/test'

transform = transforms.Compose([
    # transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader_128 = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader_128 = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f'Number of training samples: {len(train_dataset)}')
print(f'Number of testing samples: {len(test_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Class names: {train_dataset.classes}')
print(f'Example image shape: {train_dataset[0][0].shape}')

Number of training samples: 50000
Number of testing samples: 10000
Number of classes: 100
Class names: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'te

### 3. 모델 정의


In [3]:
import torch.nn as nn

class MyCustomBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride, activation=nn.ReLU, norm_layer=nn.BatchNorm2d):
        super(MyCustomBlock,self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,stride,1),
            norm_layer(out_channels),
            activation()
        )

    def forward(self,x):
        out = self.conv(x)

        return out


class MyCustomNet(nn.Module):
    def __init__(self,a=1, activation=nn.ReLU, norm_layer=nn.BatchNorm2d):
        super(MyCustomNet,self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32*a,3,2,1),
            norm_layer(32*a),
            activation()
        )

        self.Mobile = nn.Sequential(
            MyCustomBlock(32*a,64,1, activation, norm_layer),
            MyCustomBlock(64,64,2, activation, norm_layer),
            MyCustomBlock(64,128,1, activation, norm_layer),
            MyCustomBlock(128,128,2, activation, norm_layer),
            MyCustomBlock(128,256,1, activation, norm_layer),
            MyCustomBlock(256,256,2, activation, norm_layer),
            MyCustomBlock(256,256,1, activation, norm_layer),
            nn.AdaptiveAvgPool2d(1)
        )

        self.dropout = nn.Dropout(0.5)

        self.FC = nn.Sequential(
            nn.Linear(256,100)
        )

    def forward(self,x):
        out = self.conv1(x)
        out = self.Mobile(out)
        out = self.dropout(out)
        out = out.view(out.size(0),-1)
        out = self.FC(out)

        return out

In [4]:
summary(MyCustomNet().to(device), (3,32,32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 16, 16]             896
       BatchNorm2d-2           [-1, 32, 16, 16]              64
              ReLU-3           [-1, 32, 16, 16]               0
            Conv2d-4           [-1, 64, 16, 16]          18,496
       BatchNorm2d-5           [-1, 64, 16, 16]             128
              ReLU-6           [-1, 64, 16, 16]               0
     MyCustomBlock-7           [-1, 64, 16, 16]               0
            Conv2d-8             [-1, 64, 8, 8]          36,928
       BatchNorm2d-9             [-1, 64, 8, 8]             128
             ReLU-10             [-1, 64, 8, 8]               0
    MyCustomBlock-11             [-1, 64, 8, 8]               0
           Conv2d-12            [-1, 128, 8, 8]          73,856
      BatchNorm2d-13            [-1, 128, 8, 8]             256
             ReLU-14            [-1, 12

### 4. train, test 함수 정의

In [5]:
def train(dataloader , model , loss_fn , optimizer , lr_scheduler):
    size = 0
    num_batches = len(dataloader)
    model.train()
    epoch_loss , epoch_correct = 0 , 0

    for i ,(data_ , target_) in enumerate(dataloader):
        data_ , target_ = data_.to(device), target_.to(device)
        optimizer.zero_grad()

        output_ = model(data_)
        
        loss = loss_fn(output_, target_)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        pred = output_.argmax(dim=1)
        correct = (pred == target_).sum().item()
        epoch_correct += correct
        epoch_loss += loss.item()
        size += len(data_)

    train_acc = epoch_correct/size
    lr_scheduler.step()

    return train_acc , epoch_loss / num_batches

In [6]:
def test(dataloader , model , loss_fn):
    size = 0
    num_baches = len(dataloader)
    epoch_loss , epoch_correct= 0 ,0
    with torch.no_grad(): # grad 연산 X
        model.eval() # evaluation dropout 연산시
        for i, (data_ , target_) in enumerate(dataloader):

            data_ , target_ = data_.to(device), target_.to(device)
            output_ = model(data_)
            loss = loss_fn(output_, target_)

            pred = output_.argmax(dim=1)
            correct = (pred == target_).sum().item()
            epoch_correct += correct
            epoch_loss += loss.item()
            size += len(data_)

    test_acc = epoch_correct/size

    return test_acc  , epoch_loss / num_baches

### 5. 학습 및 테스트

In [7]:
EPOCHS = 30

batchsize_train_logs = { "batch_norm_acc":[],
                        "group_norm_acc":[],
                        }
batchsize_test_logs = { "batch_norm_acc":[],
                        "group_norm_acc":[],
                        }

gn_layer = lambda channels: nn.GroupNorm(num_groups=16, num_channels=channels)
models = {
    "batch_norm_2": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_2": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_4": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_4": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_8": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_8": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_16": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_16": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_32": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_32": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_64": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_64": MyCustomNet(norm_layer=gn_layer).to(device),
    "batch_norm_128": MyCustomNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm_128": MyCustomNet(norm_layer=gn_layer).to(device),
}
models_name = list(models.keys())
criterion = nn.CrossEntropyLoss()

In [ ]:
# batch_size별 모델 학습
batchsize_test_logs_name = list(batchsize_test_logs.keys())
iteration = 0
for iteration in range(7):
    train_loader = DataLoader(train_dataset, batch_size=2**(iteration+1), shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=2**(iteration+1), shuffle=False)

    # group norm 학습
    current_model = models["group_norm_" + str(2**(iteration+1))]
    optimizer = optim.SGD(current_model.parameters(), 1e-3, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    print('='*50)
    print(f'current_batch_size: {2**(iteration+1)} - group_norm')
    for epoch in tqdm(range(EPOCHS)):
        train_acc, train_loss = train(train_loader, current_model, criterion, optimizer, scheduler)
        test_acc, test_loss = test(test_loader, current_model, criterion)

        print(f'groupnorm_train_acc:{train_acc:.4f} groupnorm_test_acc:{test_acc:.4f}')
    
    batchsize_train_logs["group_norm_acc"].append(train_acc)
    batchsize_test_logs["group_norm_acc"].append(test_acc)

    # batch norm 학습
    current_model = models["batch_norm_" + str(2**(iteration+1))]
    optimizer = optim.SGD(current_model.parameters(), 1e-3, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    print('='*50)
    print(f'current_batch_size: {2**(iteration+1)} - batch_norm')
    for epoch in tqdm(range(EPOCHS)):
        train_acc, train_loss = train(train_loader, current_model, criterion, optimizer, scheduler)
        test_acc, test_loss = test(test_loader, current_model, criterion)

        print(f'batchnorm_train_acc:{train_acc:.4f} batchnorm_test_acc:{test_acc:.4f}')
    
    batchsize_train_logs["batch_norm_acc"].append(train_acc)
    batchsize_test_logs["batch_norm_acc"].append(test_acc)

current_batch_size: 2 - group_norm


  3%|▎         | 1/30 [02:35<1:15:05, 155.37s/it]

groupnorm_train_acc:0.0587 groupnorm_test_acc:0.1148



  7%|▋         | 2/30 [05:09<1:12:03, 154.40s/it]

groupnorm_train_acc:0.1242 groupnorm_test_acc:0.1785



...

batchnorm_train_acc:0.1476 batchnorm_test_acc:0.2064



 10%|█         | 3/30 [01:31<13:41, 30.43s/it]

batchnorm_train_acc:0.1993 batchnorm_test_acc:0.2328



 13%|█▎        | 4/30 [02:01<13:08, 30.31s/it]

batchnorm_train_acc:0.2486 batchnorm_test_acc:0.2726



### 7. 시각화

In [ ]:
import matplotlib.pyplot as plt

# batch size별 모델 정확도 시각화
plt.figure(figsize=(10, 6))
batchsize__ = [1, 2, 3, 4, 5, 6, 7]

plt.plot(batchsize__, batchsize_test_logs["batch_norm_acc"], 'b-o', label='Batch Norm Accuracy')
plt.plot(batchsize__, batchsize_test_logs["group_norm_acc"], 'r-s', label='Group Norm Accuracy')
plt.title(f'Model Accuracy by Batch Size', fontsize=15)
plt.xlabel('Batch Size', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.legend()

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# batch size별 모델 정확도 시각화
plt.figure(figsize=(10, 6))
batchsize__ = [1, 2, 3, 4, 5, 6, 7]

plt.plot(batchsize__, batchsize_train_logs["batch_norm_acc"], 'b-o', label='Batch Norm Accuracy')
plt.plot(batchsize__, batchsize_train_logs["group_norm_acc"], 'r-s', label='Group Norm Accuracy')
plt.title(f'Model Accuracy by Batch Size', fontsize=15)
plt.xlabel('Batch Size', fontsize=12)
plt.ylabel('Train Accuracy', fontsize=12)
plt.legend()

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()